# Face Recognition: My 4 Friends, a Classifier, and Face Verification

The classic version of this project is a CNN that recognises my four friends (Dong, Aum, TJ and Mark) from one graduation photo each. Here I build it, then ask the question it skips: **what does it say about everyone else?** Then I build what real systems do: turn every face into a vector, and decide "same person or not" by distance, so my friends can be recognised from a single photo and strangers get "unknown".

**Data**
- **My friends** → [friends/](https://github.com/Pisit-Janthawee/dl-multiclassification-face-recognition/blob/main/friends) · 4 graduation photos, one per person, 512 × 512 (for enrollment and training)
- **My friends, unseen** → [rand_images/](https://github.com/Pisit-Janthawee/dl-multiclassification-face-recognition/blob/main/rand_images) · new photos of the same four people from another day and place, with head turns from frontal to profile (the real test)
- **Strangers** → [Labeled Faces in the Wild (LFW)](https://www.kaggle.com/datasets/jessicali9530/lfw-dataset) · 13,233 news photos of 5,749 public figures, to play the strangers and to measure "same or different?" on people the model never saw

## Aim and objectives

[![overview](https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/assets/png/face_overview.png)](https://github.com/Pisit-Janthawee/dl-multiclassification-face-recognition/blob/main/assets/png/face_overview.png)

**Aim** → recognise faces the way a phone or a passport gate does: compare two faces and decide whether they belong to the same person, including people the model never saw in training.

**Objectives**
1. Build the classic closed-set classifier (Eigenfaces + SVM, a small CNN) for my 4 friends
2. Show what it does with strangers
3. Turn faces into embeddings and compare a face-trained model with a generic one
4. Measure verification honestly: unseen people, no near-duplicate photos, a threshold set on separate people
5. Recognise my friends from one photo each on **new photos** they never saw, and say "unknown" to everyone else
6. Check which parts of the face the decision rests on

**Where it could help** → unlocking a phone, checking a passport photo against the traveller, or tagging people in a photo library. All of these compare a new face with one enrolled photo and need a "no match" answer; none of them can retrain a classifier every time a new person shows up. The same tools can also be misused for surveillance, which is why consent and error rates matter as much as accuracy.

# [1/10] Setup

**What do I need before I start?**

Every setting lives in the next cell. Nothing is written to disk and no app starts unless you turn it on there.

In [ ]:
"""Optional · Add Input → dataset jessicali9530/lfw-dataset, Accelerator GPU, Internet On, then Run All. Explanations: README.md

[1/20] DATA_DIR: folder searched (recursively) for the LFW person folders; if none is found, LFW is downloaded with scikit-learn
    Default = '/kaggle/input'
[2/20] WORK_DIR: folder for the downloaded data, figures and the app settings
    Default = '/kaggle/working'
[3/20] FRIENDS: the four people of the classifier (photo file = name in lower case + .png)
    Default = ['Dong', 'Aum', 'TJ', 'Mark']
[4/20] FRIENDS_DIR: local folder with my friends' photos; if they are not there, they are downloaded from FRIENDS_URL
    Default = 'friends'
[5/20] FRIENDS_URL: where the photos live in the GitHub repository
    Default = 'https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/friends'
[6/20] N_AUG: augmented copies made from each friend's single photo, for train / validation / test
    Default = (150, 25, 25)
[7/20] REAL_DIR: folder with new, unseen photos of my friends (one sub-folder per friend); if missing, downloaded from REAL_URL
    Default = 'rand_images'
[8/20] REAL_URL: where the unseen photos live in the GitHub repository (with a manifest.json listing them)
    Default = 'https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/rand_images'
[9/20] CROP: face box (left, top, right, bottom) inside the 250 x 250 LFW photo
    Default = (60, 60, 190, 190)
[10/20] IMG_SIZE: face size fed to the face model (FaceNet expects 160)
    Default = 160
[11/20] CNN_SIZE: face size for the small CNN
    Default = 96
[12/20] CNN_EPOCHS: training epochs of the small CNN
    Default = 30
[13/20] N_PAIRS: same-person pairs and different-person pairs, each, for validation and for test
    Default = 3000
[14/20] TARGET_FAR: share of different-person pairs the threshold may wrongly accept (false accept rate)
    Default = 0.001
    For example:
        TARGET_FAR = 0.01    (a photo-tagging app)
        TARGET_FAR = 0.0001  (a door lock)
[15/20] DUP_HAMMING: max differing bits (of 256) for two photos of one person to count as near-duplicates
    Default = 6
[16/20] BATCH_SIZE: faces per training step and per forward pass
    Default = 32
[17/20] SEED: random seed
    Default = 42
[18/20] DISPLAY_SEED: which faces are shown in the example grids (None = a new random sample on every run)
    Default = None
[19/20] SAVE_FIGURES: write every chart to WORK_DIR/assets/png
    Default = False
[20/20] LAUNCH_APP: start the Gradio app at the end (public link on Kaggle)
    Default = False
"""
DATA_DIR          = '/kaggle/input'
WORK_DIR          = '/kaggle/working'
FRIENDS           = ['Dong', 'Aum', 'TJ', 'Mark']
FRIENDS_DIR       = 'friends'
FRIENDS_URL       = 'https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/friends'
N_AUG             = (150, 25, 25)
REAL_DIR          = 'rand_images'
REAL_URL          = 'https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/rand_images'
CROP              = (60, 60, 190, 190)
IMG_SIZE          = 160
CNN_SIZE          = 96
CNN_EPOCHS        = 30
N_PAIRS           = 3000
TARGET_FAR        = 0.001
DUP_HAMMING       = 6
BATCH_SIZE        = 32
SEED              = 42
DISPLAY_SEED      = None
SAVE_FIGURES      = False
LAUNCH_APP        = False

PAD = 26

In [ ]:
%pip install -q --no-deps facenet-pytorch
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageOps
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

from facenet_pytorch import MTCNN, InceptionResnetV1
import torchvision

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True     # same CNN result on every run of the same GPU
torch.backends.cudnn.benchmark = False
rng = np.random.default_rng(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
sns.set_theme(style='whitegrid')
GREY, RED = '#9e9e9e', '#c62828'
ASSETS = Path(WORK_DIR) / 'assets' / 'png'
if SAVE_FIGURES:
    ASSETS.mkdir(parents=True, exist_ok=True)

In [ ]:
def show_block(title, items):
    print(f'{title}:')
    for key, value in items.items():
        print(f'    {key:<{PAD}}{value}')


def finish(fig, name):
    """Show a figure; with SAVE_FIGURES also write it to WORK_DIR/assets/png."""
    if SAVE_FIGURES:
        fig.savefig(ASSETS / f'{name}.png', bbox_inches='tight', dpi=110)
    plt.show()


show_rng = np.random.default_rng(DISPLAY_SEED)   # only decides which faces are displayed
show_block('Device', {'torch': torch.__version__, 'device': DEVICE})

# [2/10] Data

**Who are my four friends, and who plays the strangers?**

**What I'm trying to do** → load my friends' photos and find their faces, then load LFW, the public faces that play everyone else.

In [ ]:
friends_dir = Path(FRIENDS_DIR)
if not all((friends_dir / f'{n.lower()}.png').exists() for n in FRIENDS):        # on Kaggle: download from the repository
    friends_dir = Path(WORK_DIR) / 'friends'
    friends_dir.mkdir(parents=True, exist_ok=True)
    for n in FRIENDS:
        urllib.request.urlretrieve(f'{FRIENDS_URL}/{n.lower()}.png', friends_dir / f'{n.lower()}.png')
friend_photos = {n: Image.open(friends_dir / f'{n.lower()}.png').convert('RGB') for n in FRIENDS}

detector = MTCNN(image_size=IMG_SIZE, margin=40, post_process=False, device=DEVICE)   # finds and crops the face
friend_faces, friend_conf = {}, {}
for n, photo in friend_photos.items():
    box, prob = detector.detect(photo)
    assert box is not None, f'(!) no face found in {n}'
    friend_faces[n] = Image.fromarray(detector(photo).permute(1, 2, 0).numpy().astype(np.uint8))
    friend_conf[n] = float(prob[0])
show_block('Friends', {n: f'photo {friend_photos[n].size[0]} x {friend_photos[n].size[1]}, face found (confidence {friend_conf[n]:.4f})' for n in FRIENDS})

In [ ]:
fig, axes = plt.subplots(2, len(FRIENDS), figsize=(3.4 * len(FRIENDS), 7))
for k, n in enumerate(FRIENDS):
    axes[0, k].imshow(friend_photos[n])
    axes[0, k].set_title(n, fontweight='bold')
    axes[1, k].imshow(friend_faces[n])
    axes[1, k].set_title(f'face found by MTCNN ({IMG_SIZE} x {IMG_SIZE})', fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('My four friends: one graduation photo each, and the face the detector cuts out', fontweight='bold')
plt.tight_layout()
finish(fig, '02_friends')

**Question:** What data do I have for my friends?

**Answer:** One photo per person, all from the same studio session: white background, the same gown, the same light.
- MTCNN (a small face detector) finds every face with confidence ≥ 0.9999 and crops it to 160 × 160
- one photo per person is exactly the situation of a phone or a door lock: you enroll once
- In short → great for enrollment, thin for training a classifier; the "training set" can only be copies of these 4 photos

**My friends, unseen** → new photos of the same four people, taken on another day and place, with the head turned from frontal to full profile. The photos show the whole upper body, so each face has to be **found, cropped and resized** first.

**Face crop in one line** → MTCNN (three small CNNs) finds the face box and 5 landmarks (eyes, nose, mouth corners); the box plus a margin is cut out and resized to 160 × 160, exactly like the enrollment photos.
- **Rotation first** → camera photos are often saved sideways with a "rotate me" tag (EXIF orientation); without applying it, the detector misses the face or crops it lying on its side
- **Head turn** → from the landmarks: how far the nose sits from the middle of the eyes, measured in eye distances (0 = frontal, about 0.5 or more = side view)
- In short → the same crop for every photo, so the models compare faces, not framing

In [ ]:
IMAGE_TYPES = {'.jpg', '.jpeg', '.png'}
real_dir = Path(REAL_DIR)
if not real_dir.exists():                                                   # on Kaggle: download from the repository
    real_dir = Path(WORK_DIR) / 'rand_images'
    manifest = json.loads(urllib.request.urlopen(f'{REAL_URL}/manifest.json').read().decode('utf-8'))
    for n, files in manifest.items():
        (real_dir / n).mkdir(parents=True, exist_ok=True)
        for f in files:
            urllib.request.urlretrieve(f'{REAL_URL}/{urllib.parse.quote(n)}/{urllib.parse.quote(f)}', real_dir / n / f)

real_rows = []
for k, n in enumerate(FRIENDS):
    for f in sorted(p for p in (real_dir / n).iterdir() if p.suffix.lower() in IMAGE_TYPES):
        photo = ImageOps.exif_transpose(Image.open(f)).convert('RGB')     # camera photos are often stored sideways + a rotate tag
        photo.thumbnail((1600, 1600))                                       # 6000 x 4000 camera files → faster detection, same face
        boxes, probs, marks = detector.detect(photo, landmarks=True)
        if boxes is None:
            real_rows.append({'friend': n, 'target': k, 'file': f.name, 'face': None, 'turn': np.nan, 'conf': 0.0})
            continue
        best = int(np.argmax([float(p) for p in probs]))
        le, re_, nose = (np.asarray(marks[best][j], float) for j in range(3))
        turn = abs(nose[0] - (le[0] + re_[0]) / 2) / max(abs(re_[0] - le[0]), 1.0)
        face = detector.extract(photo, np.asarray([boxes[best]], float), None)
        real_rows.append({'friend': n, 'target': k, 'file': f.name, 'face': Image.fromarray(face.permute(1, 2, 0).numpy().astype(np.uint8)),
                          'turn': turn, 'conf': float(probs[best])})
real_df = pd.DataFrame(real_rows)
real_df['head'] = pd.cut(real_df['turn'], [-0.01, 0.15, 0.5, np.inf], labels=['frontal', 'turned', 'side'])
real_ok = real_df[real_df.face.notna()].reset_index(drop=True)
display(pd.crosstab(real_df.friend, real_df['head'].cat.add_categories('no face found').fillna('no face found')).reindex(FRIENDS))

In [ ]:
picks = real_ok.groupby('friend', sort=False).apply(lambda g: g.sort_values('turn').iloc[np.linspace(0, len(g) - 1, min(6, len(g))).astype(int)]).reset_index(drop=True)
fig, axes = plt.subplots(len(FRIENDS), 6, figsize=(13, 2.5 * len(FRIENDS)))
for ax in axes.ravel():
    ax.axis('off')
for r, n in enumerate(FRIENDS):
    for c, (_, row) in enumerate(picks[picks.friend == n].iterrows()):
        axes[r, c].imshow(row.face)
        axes[r, c].set_title(f"{n} · {row['head']} ({row.turn:.2f})", fontsize=8.5)
plt.suptitle('Unseen photos of my friends after face crop + resize (sorted from frontal to side view)', fontweight='bold')
plt.tight_layout()
finish(fig, '02_unseen_faces')

**Question:** Does the face crop work on the unseen photos?

**Answer:** Yes, for all 88 photos, once the camera rotation is applied (without it, 15 of the 6000 × 4000 camera photos had no face found, and others were cropped lying on their side).
- 55 frontal, 19 turned, 14 side views
- Mark has the most photos (40 of 88), Aum the fewest (10)
- In short → every unseen photo now looks like an enrollment crop: a 160 × 160 face

**The strangers: LFW** → news photos of public figures, one folder per person.

In [ ]:
def find_lfw(*roots, min_people=1000):
    """First folder holding at least `min_people` person folders with *_0001.jpg photos (Kaggle mount paths differ)."""
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for first in root.rglob('*_0001.jpg'):
            base = first.parent.parent
            if sum(1 for p in base.iterdir() if p.is_dir()) >= min_people:
                return base
    return None


lfw_dir = find_lfw(DATA_DIR, WORK_DIR)
if lfw_dir is None:                               # not mounted → download with scikit-learn (needs Internet)
    from sklearn.datasets import fetch_lfw_people
    fetch_lfw_people(data_home=str(Path(WORK_DIR) / 'lfw_data'), download_if_missing=True, min_faces_per_person=1)
    lfw_dir = find_lfw(Path(WORK_DIR) / 'lfw_data')
assert lfw_dir is not None, '(!) LFW not found: add the dataset jessicali9530/lfw-dataset or turn Internet on'

faces_df = pd.DataFrame([{'person': p.name, 'path': str(f)} for p in sorted(lfw_dir.iterdir()) if p.is_dir()
                         for f in sorted(p.glob('*.jpg'))])
photos = faces_df['person'].value_counts()
faces_df['n_photos'] = faces_df['person'].map(photos)
show_block('Outputs', {'LFW folder': lfw_dir, 'photos': f'{len(faces_df):,}', 'people': f'{len(photos):,}'})

In [ ]:
display(pd.DataFrame({'people': [(photos == 1).sum(), ((photos >= 2) & (photos < 10)).sum(), ((photos >= 10) & (photos < 50)).sum(), (photos >= 50).sum()]},
                     index=['1 photo', '2-9 photos', '10-49 photos', '50+ photos']))
display(photos.head(8).rename('photos'))

In [ ]:
picks = show_rng.choice(len(faces_df), 24, replace=False)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.6))
for ax, i in zip(axes.ravel(), picks):
    ax.imshow(Image.open(faces_df.path[i]))
    ax.set_title(faces_df.person[i].replace('_', ' ')[:22], fontsize=8)
    ax.axis('off')
plt.suptitle('Random LFW photos: faces "in the wild" (news photos, any light, pose and expression)', fontweight='bold')
plt.tight_layout()
finish(fig, '02_samples')

**Question:** How are the LFW photos spread over people?

**Answer:** Very unevenly.
- most people (about 4,000 of 5,749) have a **single** photo, so they can only ever appear as strangers or as one half of a "different person" pair
- a handful have hundreds: George W Bush alone has 530 photos
- In short → plenty of people, few photos each: perfect for "same or different?", poor for training a classifier per person

# [3/10] Face crop and near-duplicates

**What part of the photo is the face, and are some photos the same shot twice?**

**What I'm trying to do** → crop the face from the centre of every photo, then look for near-identical photos of the same person. A near-copy in a "same person" test pair would make verification look easier than it is.

In [ ]:
def load_face(path, size=IMG_SIZE, gray=False):
    image = Image.open(path).convert('L' if gray else 'RGB').crop(CROP).resize((size, size), Image.BILINEAR)
    return np.asarray(image)


def dhash_bits(path, hash_size=16):
    small = np.asarray(Image.open(path).convert('L').crop(CROP).resize((hash_size + 1, hash_size), Image.BILINEAR), np.int16)
    return (small[:, 1:] > small[:, :-1]).ravel()


dup_pairs = []
for person, group in tqdm(faces_df[faces_df.n_photos >= 2].groupby('person'), desc='[1.1] Near-duplicates per person'):
    idx = group.index.to_numpy()
    bits = np.stack([dhash_bits(p) for p in group.path]).astype(np.float32)
    dist = bits @ (1 - bits).T + (1 - bits) @ bits.T
    for a, b in zip(*np.where(np.triu(dist <= DUP_HAMMING, 1))):
        dup_pairs.append((idx[a], idx[b], int(dist[a, b])))
dup_set = {frozenset(p[:2]) for p in dup_pairs}
show_block('Outputs', {'near-duplicate pairs': len(dup_pairs), 'people affected': len({faces_df.person[a] for a, _, _ in dup_pairs})})

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
example = faces_df.path[show_rng.integers(len(faces_df))]
axes[0, 0].imshow(Image.open(example))
axes[0, 0].add_patch(plt.Rectangle(CROP[:2], CROP[2] - CROP[0], CROP[3] - CROP[1], fill=False, ec=RED, lw=2))
axes[0, 0].set_title('full photo + face box', fontsize=9)
axes[1, 0].imshow(load_face(example))
axes[1, 0].set_title(f'face crop {IMG_SIZE} x {IMG_SIZE}', fontsize=9)
shown = [dup_pairs[k] for k in show_rng.choice(len(dup_pairs), min(5, len(dup_pairs)), replace=False)] if dup_pairs else []
for col, (a, b, d) in enumerate(shown, start=1):
    for row, k in enumerate([a, b]):
        axes[row, col].imshow(load_face(faces_df.path[k]))
        axes[row, col].set_title(f"{faces_df.person[k].replace('_', ' ')[:18]}\n{Path(faces_df.path[k]).name[-8:]}  d = {d}", fontsize=8)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Left: the face crop. Right: near-duplicate pairs (same person, almost the same photo)', fontweight='bold')
plt.tight_layout()
finish(fig, '03_crop_duplicates')

**Question:** Can the same photo hide twice in LFW?

**Answer:** Almost never: only 1 near-identical pair in the whole dataset (the same news photo saved twice).
- LFW was collected with care, unlike many scraped face sets
- a near-copy can only appear in a **same-person** pair, where it would make verification look too easy
- so every "same person" test pair later skips it anyway

**Difference hash in one line** → shrink the face to 17 × 16 grey pixels and ask 256 times "is this pixel brighter than its right neighbour?"; two photos whose answers differ in at most `DUP_HAMMING` places are near-copies.
- In short → a fingerprint that survives resizing and small brightness changes, but not a different pose

# [4/10] The classic approach: a classifier for my 4 friends

**Can a model learn to tell my 4 friends apart?**

**What I'm trying to do** → rebuild the original project: a classifier that picks one of Dong, Aum, TJ or Mark. With one photo each, the training images can only be **augmented copies**: small random rotations, shifts, zooms, brightness changes and mirror flips of that one photo, as in the original notebook. Train, validation and test copies are made with different random draws, but they all come from the same 4 photos.

In [ ]:
augmenter = torchvision.transforms.Compose([
    torchvision.transforms.RandomAffine(degrees=10, translate=(0.06, 0.06), scale=(0.9, 1.1), fill=255),
    torchvision.transforms.ColorJitter(brightness=0.25, contrast=0.25),
    torchvision.transforms.RandomHorizontalFlip()])
known = FRIENDS
cls_faces, y_cls, split = [], [], []
for s_name, n_copies, seed in zip(['train', 'validation', 'test'], N_AUG, [SEED, SEED + 1, SEED + 2]):
    torch.manual_seed(seed)
    for k, n in enumerate(FRIENDS):
        for _ in range(n_copies):
            cls_faces.append(augmenter(friend_faces[n]))
            y_cls.append(k)
            split.append(s_name)
y_cls, split = np.array(y_cls), np.array(split)
tr_idx, va_idx, te_idx = (np.flatnonzero(split == s_name) for s_name in ['train', 'validation', 'test'])
torch.manual_seed(SEED)
display(pd.crosstab(pd.Series(y_cls).map(dict(enumerate(FRIENDS))).rename('friend'), pd.Series(split, name='split'))[['train', 'validation', 'test']])

In [ ]:
fig, axes = plt.subplots(len(FRIENDS), 8, figsize=(14, 7.4))
for k, n in enumerate(FRIENDS):
    for j, i in enumerate(tr_idx[y_cls[tr_idx] == k][:8]):
        axes[k, j].imshow(cls_faces[i])
        axes[k, j].axis('off')
    axes[k, 0].set_title(n, loc='left', fontweight='bold')
plt.suptitle('Training copies: every image of a person is the same photo, slightly rotated, shifted, zoomed, re-lit or mirrored', fontweight='bold')
plt.tight_layout()
finish(fig, '04_augmented')

# **Eigenfaces in one line** → describe every face as "the average face plus a little of each typical way faces differ", then let an SVM separate people by those few numbers.
#
# **How, in 3 steps**
# 1. Flatten every 64 × 64 grey face into 4,096 numbers
# 2. **PCA** finds the directions in which faces vary most (light from the left, smile, glasses …); each direction, drawn as a picture, is an "eigenface"
# 3. Keep the first 100 and train an **SVM** on them
#
# $$x \approx \mu + \sum_{k=1}^{100} z_k\, u_k$$
# - $x$ → one face as 4,096 pixel values, $\mu$ → the average face
# - $u_k$ → the k-th eigenface, $z_k$ → how much of it this face has (the numbers the SVM sees)
# - In short → 4,096 pixels squeezed into 100 "face ingredients"

In [ ]:
X_gray = np.stack([np.asarray(f.convert('L').resize((64, 64))).ravel() for f in cls_faces]).astype(np.float32) / 255
pca = PCA(n_components=100, whiten=True, random_state=SEED).fit(X_gray[tr_idx])
svm = GridSearchCV(SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=SEED),
                   {'C': [1, 10, 100], 'gamma': ['scale', 0.001, 0.01]}, cv=3).fit(pca.transform(X_gray[tr_idx]), y_cls[tr_idx])
acc_svm = accuracy_score(y_cls[te_idx], svm.predict(pca.transform(X_gray[te_idx])))
show_block('Eigenfaces + SVM', {'best params': svm.best_params_, 'test accuracy': f'{acc_svm:.3f}'})

In [ ]:
fig, axes = plt.subplots(1, 9, figsize=(18, 2.6))
axes[0].imshow(pca.mean_.reshape(64, 64), cmap='gray')
axes[0].set_title('average face')
for k, ax in enumerate(axes[1:]):
    ax.imshow(pca.components_[k].reshape(64, 64), cmap='gray')
    ax.set_title(f'eigenface {k + 1}')
for ax in axes:
    ax.axis('off')
plt.suptitle('The average of all training copies and the first 8 eigenfaces (directions in which they vary most)', fontweight='bold')
plt.tight_layout()
finish(fig, '04_eigenfaces')

**Small CNN in one line** → four convolution blocks learn edges, then eyes and mouths, then a 4-way softmax picks the person.

- **Input** → 96 × 96 colour face, normalised with the training mean and spread; random left-right flips as augmentation (a mirrored face is still a valid face, unlike a mirrored chest X-ray)
- **Keep the layout** → the last feature map (6 × 6) is flattened, not averaged into one number per channel: *where* the eyes and nose sit matters for a face
- **Output** → 4 probabilities that **always sum to 1**: the model has no way to say "none of them"
- In short → the same idea as the original project, with a slightly stronger network

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 128))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(128 * (CNN_SIZE // 16) ** 2, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def to_tensor(arr):
    return torch.from_numpy(np.asarray(arr, np.float32) / 255).permute(0, 3, 1, 2)


X_cnn = to_tensor(np.stack([np.asarray(f.resize((CNN_SIZE, CNN_SIZE))) for f in cls_faces]))
MU, SD = X_cnn[tr_idx].mean((0, 2, 3), keepdim=True), X_cnn[tr_idx].std((0, 2, 3), keepdim=True)   # training photos only
X_cnn = (X_cnn - MU) / SD
cnn = SmallCNN(len(known)).to(DEVICE)
opt = torch.optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
w = torch.tensor(len(tr_idx) / (len(known) * np.bincount(y_cls[tr_idx])), dtype=torch.float32, device=DEVICE)
history = []
for epoch in tqdm(range(CNN_EPOCHS), desc='[1.1] Training the CNN'):
    cnn.train()
    for b in torch.from_numpy(rng.permutation(tr_idx)).split(BATCH_SIZE):
        xb, yb = X_cnn[b].to(DEVICE), torch.from_numpy(y_cls[b.numpy()]).to(DEVICE)
        flip = torch.rand(len(xb), device=DEVICE) < 0.5
        xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
        loss = F.cross_entropy(cnn(xb), yb, weight=w)
        opt.zero_grad()
        loss.backward()
        opt.step()
    cnn.eval()
    with torch.no_grad():
        va_pred = cnn(X_cnn[va_idx].to(DEVICE)).argmax(1).cpu().numpy()
    history.append({'epoch': epoch + 1, 'train loss': loss.item(), 'validation accuracy': accuracy_score(y_cls[va_idx], va_pred)})


def cnn_proba(x):
    cnn.eval()
    with torch.no_grad():
        return torch.cat([F.softmax(cnn(x[i:i + 256].to(DEVICE)), 1).cpu() for i in range(0, len(x), 256)]).numpy()


p_te = cnn_proba(X_cnn[te_idx])
acc_cnn = accuracy_score(y_cls[te_idx], p_te.argmax(1))
show_block('Small CNN', {'final validation accuracy': f"{history[-1]['validation accuracy']:.3f}", 'test accuracy': f'{acc_cnn:.3f}'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
pd.DataFrame(history).set_index('epoch')['validation accuracy'].plot(ax=axes[0], color='black', marker='o')
axes[0].set(title='Small CNN: validation accuracy per epoch', ylim=(0, 1.02))
ConfusionMatrixDisplay(confusion_matrix(y_cls[te_idx], p_te.argmax(1)), display_labels=FRIENDS).plot(ax=axes[1], cmap='Greys', colorbar=False)
axes[1].set_title(f'Small CNN on the test photos (accuracy {acc_cnn:.2f})')
axes[1].grid(False)
plt.tight_layout()
finish(fig, '04_classifier')

**Question:** Can a model tell my 4 friends apart?

**Answer:** Perfectly, on paper: Eigenfaces + SVM and the small CNN both get **100 %** of the test copies right.
- but every test image is a rotated, shifted or re-lit copy of the **same photo** the model trained on, so this only measures "recognise this photo again"
- the real test is the next cell: new photos of the same four friends, from another day and place
- In short → a 4-person classifier is easy to build and easy to score high on copies

**Question:** And on new photos of my friends?

The 88 unseen photos from [2/10]: another day and place, a shirt and tie instead of a gown, and the head turned from frontal to profile.

In [ ]:
X_real_gray = np.stack([np.asarray(f.convert('L').resize((64, 64))).ravel() for f in real_ok.face]).astype(np.float32) / 255
X_real_cnn = (to_tensor(np.stack([np.asarray(f.resize((CNN_SIZE, CNN_SIZE))) for f in real_ok.face])) - MU) / SD
real_ok['svm'] = svm.predict(pca.transform(X_real_gray))
p_real = cnn_proba(X_real_cnn)
real_ok['cnn'] = p_real.argmax(1)
acc_real = pd.DataFrame({'Eigenfaces + SVM': real_ok.assign(ok=real_ok.svm == real_ok.target).groupby('head', observed=True).ok.mean(),
                         'small CNN': real_ok.assign(ok=real_ok.cnn == real_ok.target).groupby('head', observed=True).ok.mean()}).T
acc_real['all photos'] = [(real_ok.svm == real_ok.target).mean(), (real_ok.cnn == real_ok.target).mean()]
display(acc_real.round(3))

**Answer:** No. On new photos the CNN is right on **53 %** and Eigenfaces + SVM on **33 %**, after 100 % on the copies.
- chance is 25 %, and always answering "Mark" (40 of the 88 photos) would already get 45 %
- both models learned the 4 graduation photos (the gown, the white studio, that exact light), not the 4 faces
- In short → 100 % on augmented copies said nothing about recognising my friends

# [5/10] The catch: what does it say about a stranger?

**Show it someone it has never seen: does it say "I don't know"?**

**I suspect** → it can't. Softmax probabilities always add up to 1, so every face gets assigned to Dong, Aum, TJ or Mark, often with high confidence.

In [ ]:
strangers = faces_df.sample(1000, random_state=SEED).reset_index(drop=True)                      # LFW public figures
p_strangers = cnn_proba((to_tensor(np.stack([load_face(p, CNN_SIZE) for p in strangers.path])) - MU) / SD)
conf = pd.DataFrame({'max probability': np.r_[p_te.max(1), p_strangers.max(1)],
                     'who': ['friend (test copy)'] * len(p_te) + ['stranger'] * len(p_strangers)})
share_confident = (p_strangers.max(1) >= 0.9).mean()
named_as = pd.Series(p_strangers.argmax(1)).map(dict(enumerate(FRIENDS))).value_counts(normalize=True).reindex(FRIENDS).fillna(0)
show_block('Strangers', {'stranger photos': len(strangers), 'assigned to one of the 4 names': '100 %',
                         'with probability >= 0.9': f'{share_confident:.1%}',
                         **{f'named "{n}"': f'{v:.1%}' for n, v in named_as.items()}})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={'width_ratios': [1, 1.4]})
sns.histplot(data=conf, x='max probability', hue='who', bins=25, stat='probability', common_norm=False, ax=axes[0],
             palette={'friend (test copy)': GREY, 'stranger': RED}, element='step')
axes[0].set_title('How sure the classifier is (highest of its 4 probabilities)')
order = np.argsort(-p_strangers.max(1))[:6]
axes[1].axis('off')
for j, k in enumerate(order):
    ax = fig.add_axes([0.43 + j * 0.095, 0.25, 0.085, 0.5])
    ax.imshow(load_face(strangers.path[k]))
    ax.set_title(f"stranger: {strangers.person[k].split('_')[-1]}\n→ \"{known[p_strangers[k].argmax()].split('_')[-1]}\" {p_strangers[k].max():.0%}", fontsize=8.5)
    ax.axis('off')
axes[1].set_title('Strangers (LFW) the classifier was most sure about, and which friend it called them', pad=28)
finish(fig, '05_strangers')

**Question:** What does the classifier say about someone it has never seen?

**Answer:** It names every stranger as one of my friends, and mostly as the same one: **88 % of the 1,000 strangers become "Mark"**, and for 84 % it is at least 90 % sure.
- with one photo per person the model learns whatever sets each photo apart; whatever separates Mark's photo from the other three happens to match most news photos, men and women alike
- a closed-set classifier has no "nobody" option, so high accuracy on its own test set says nothing about the real world
- adding a person means collecting their photos and **retraining**
- In short → this is classification over a fixed list, not recognition. Real systems compare faces instead

[![closed set vs open set](https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/assets/png/m_closed_vs_open.png)](https://github.com/Pisit-Janthawee/dl-multiclassification-face-recognition/blob/main/assets/png/m_closed_vs_open.png)

# [6/10] Face embeddings: one vector per face

**Can a model turn any face into a "face fingerprint"?**

**What I'm trying to do** → map every face to a vector so that photos of the same person land close together and different people land far apart, even for people the model never saw. I compare two models:
- **FaceNet** (Inception-ResNet, trained on VGGFace2: 3.3 million photos of 9,000 people, none of them used here to fit anything) → 512 numbers per face
- **ResNet-18 trained on ImageNet** (objects, not faces) → 512 numbers per face, as a "generic features" baseline

**Face embedding in one line** → a network trained on thousands of people learns which face details tell people apart (eye distance, nose shape) and which don't (light, smile); its output is a short vector that keeps only the first kind.

$$\text{similarity}(a, b) = \cos\theta = \frac{e_a \cdot e_b}{\lVert e_a\rVert\,\lVert e_b\rVert}$$
- $e_a, e_b$ → embeddings of two faces; 1 = same direction (same person), 0 or below = unrelated
- In short → same person = small angle between the vectors

In [ ]:
def facenet_input(paths):
    return torch.from_numpy((np.stack([load_face(p) for p in paths]).astype(np.float32) - 127.5) / 128).permute(0, 3, 1, 2)


IMAGENET_MEAN, IMAGENET_STD = torch.tensor([0.485, 0.456, 0.406])[:, None, None], torch.tensor([0.229, 0.224, 0.225])[:, None, None]


def resnet_input(paths):
    return (to_tensor(np.stack([load_face(p, 224) for p in paths])) - IMAGENET_MEAN) / IMAGENET_STD


facenet = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
resnet = torchvision.models.resnet18(weights='IMAGENET1K_V1')
resnet.fc = nn.Identity()
resnet = resnet.eval().to(DEVICE)


def embed(model, make_input, paths, desc):
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(paths), BATCH_SIZE), desc=desc):
            out.append(F.normalize(model(make_input(paths[i:i + BATCH_SIZE]).to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy()


need = faces_df.index[faces_df.n_photos >= 2].union(faces_df.index[faces_df.path.isin(strangers.path)])
paths_needed = faces_df.path[need].tolist()
start = time.time()
E_face = dict(zip(need, embed(facenet, facenet_input, paths_needed, '[1.1] FaceNet embeddings')))
E_gen = dict(zip(need, embed(resnet, resnet_input, paths_needed, '[1.2] ResNet-18 embeddings')))
show_block('Outputs', {'faces embedded': f'{len(need):,}', 'vector size': 512, 'time': f'{time.time() - start:.0f} s on {DEVICE}'})

In [ ]:
map_people = photos[(photos >= 30) & (photos <= 80)].index[:8].tolist()
map_idx = faces_df.index[faces_df.person.isin(map_people)]
fig, axes = plt.subplots(1, 2, figsize=(16, 6.4))
for ax, (title, E) in zip(axes, [('FaceNet (trained on faces)', E_face), ('ResNet-18 (trained on objects)', E_gen)]):
    xy = TSNE(perplexity=20, random_state=SEED, init='pca').fit_transform(np.stack([E[i] for i in map_idx]))
    for k, person in enumerate(map_people):
        m = (faces_df.person[map_idx] == person).to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=22, color=plt.cm.tab10(k), label=person.replace('_', ' '))
    ax.set(title=f'{title}: 8 people, one dot per photo', xticks=[], yticks=[])
axes[1].legend(fontsize=8, loc='center left', bbox_to_anchor=(1, 0.5))
plt.suptitle('Embeddings drawn in 2D (t-SNE): photos of one person should form one tight group', fontweight='bold')
plt.tight_layout()
finish(fig, '06_embedding_map')

**Question:** Does a face-trained model put each person's photos together?

**Answer:** Yes for FaceNet: each person forms its own tight group. The ImageNet model mixes people up, because it learned what separates cats from cars, not one face from another.
- neither model was trained on these 8 people
- In short → the knowledge "what makes a face unique" transfers to new people; generic image features don't carry it

# [7/10] Verification on people the model has never seen

**Two photos: same person or not, and how often is the answer wrong?**

**What I'm trying to do** → my friends have one photo each, so "same person?" can't be measured on them. I measure it on LFW people instead, split by **person** into a validation half (to pick the threshold) and a test half (to score it once).

**Honest pairs, in 4 steps**
1. Take every LFW person with at least 2 photos and split the **people** 50/50 into validation and test
2. Same-person pairs → pick a person (each equally likely, so George W Bush's 530 photos don't dominate), then two of their photos, skipping near-duplicates from [3/10]
3. Different-person pairs → two photos of two different people from the same half
4. Pick the threshold on validation so that only `TARGET_FAR` of different-person pairs pass, then apply it to test unchanged

**Two kinds of error**
- **False accept (FAR)** → two different people called "same": a stranger unlocks your phone
- **False reject (FRR)** → the same person called "different": you have to type your PIN
- In short → one threshold trades one error for the other; the application decides which one hurts more

[![FAR and FRR](https://raw.githubusercontent.com/Pisit-Janthawee/dl-multiclassification-face-recognition/main/assets/png/m_far_frr.png)](https://github.com/Pisit-Janthawee/dl-multiclassification-face-recognition/blob/main/assets/png/m_far_frr.png)

In [ ]:
ver_people = np.array(sorted(set(photos.index[photos >= 2]) - set(known)))
rng.shuffle(ver_people)
halves = {'validation': set(ver_people[: len(ver_people) // 2]), 'test': set(ver_people[len(ver_people) // 2:])}


def make_pairs(people, n, seed):
    r = np.random.default_rng(seed)
    sub = faces_df[faces_df.person.isin(people)]
    groups = {p: g.index.to_numpy() for p, g in sub.groupby('person')}
    names = list(groups)
    same, diff = set(), set()
    while len(same) < n:                       # every person equally likely, so people with hundreds of photos don't dominate
        p = names[r.integers(len(names))]
        a, b = r.choice(groups[p], 2, replace=False)
        if frozenset((a, b)) not in dup_set:
            same.add((min(a, b), max(a, b)))
    while len(diff) < n:
        p, q = r.choice(len(names), 2, replace=False)
        diff.add((r.choice(groups[names[p]]), r.choice(groups[names[q]])))
    pairs = np.array(list(same) + list(diff))
    return pairs, np.r_[np.ones(len(same)), np.zeros(len(diff))]


pairs = {name: make_pairs(people, N_PAIRS, SEED + k) for k, (name, people) in enumerate(halves.items())}

# Eigenfaces as a third "embedding": PCA fitted only on validation people, so test people stay unseen
pca_ver = PCA(n_components=100, whiten=True, random_state=SEED).fit(
    np.stack([load_face(p, 64, gray=True).ravel() for p in faces_df.path[faces_df.person.isin(halves['validation'])].sample(3000, random_state=SEED)]) / 255)
pair_idx = np.unique(np.concatenate([pairs[s][0].ravel() for s in pairs]))
Z = pca_ver.transform(np.stack([load_face(p, 64, gray=True).ravel() for p in faces_df.path[pair_idx]]) / 255)
E_eig = dict(zip(pair_idx, Z / np.linalg.norm(Z, axis=1, keepdims=True)))
EMBEDDINGS = {'FaceNet': E_face, 'ResNet-18 (ImageNet)': E_gen, 'Eigenfaces (PCA)': E_eig}
show_block('Pairs', {f'{s} people': f"{len(halves[s]):,} → {int(pairs[s][1].sum()):,} same + {int((1 - pairs[s][1]).sum()):,} different" for s in pairs})

In [ ]:
def similarity(E, pr):
    return np.array([E[a] @ E[b] for a, b in pr])


rows, scores = [], {}
for name, E in EMBEDDINGS.items():
    s_val, s_te = similarity(E, pairs['validation'][0]), similarity(E, pairs['test'][0])
    y_val, y_te = pairs['validation'][1], pairs['test'][1]
    thr = np.quantile(s_val[y_val == 0], 1 - TARGET_FAR)               # only TARGET_FAR of validation impostors pass
    fpr, tpr, _ = roc_curve(y_te, s_te)
    eer = fpr[np.argmin(np.abs(fpr - (1 - tpr)))]
    scores[name] = (s_te, y_te, thr, fpr, tpr)
    rows.append({'model': name, 'test AUC': roc_auc_score(y_te, s_te), 'equal error rate': eer, 'threshold (from validation)': thr,
                 'test FAR at threshold': (s_te[y_te == 0] >= thr).mean(), 'test TAR at threshold': (s_te[y_te == 1] >= thr).mean()})
ver_table = pd.DataFrame(rows).set_index('model')
THRESHOLD = float(ver_table.loc['FaceNet', 'threshold (from validation)'])

In [ ]:
display(ver_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
s_te, y_te, thr, _, _ = scores['FaceNet']
bins = np.linspace(-0.5, 1, 61)
axes[0].hist(s_te[y_te == 0], bins=bins, density=True, histtype='stepfilled', color=GREY, alpha=0.5, label='different people')
axes[0].hist(s_te[y_te == 1], bins=bins, density=True, histtype='stepfilled', color='black', alpha=0.45, label='same person')
axes[0].axvline(thr, color=RED, lw=2, label=f'threshold {thr:.2f} (FAR {TARGET_FAR:.1%} on validation)')
axes[0].set(title='FaceNet: cosine similarity of test pairs (unseen people)', xlabel='cosine similarity')
axes[0].legend(fontsize=9)
for name, color in zip(EMBEDDINGS, ['black', RED, GREY]):
    _, _, _, fpr, tpr = scores[name]
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC {ver_table.loc[name, 'test AUC']:.3f})")
axes[1].set(xscale='log', xlim=(1e-4, 1), title='ROC on test pairs (false accept rate on a log scale)', xlabel='false accept rate (FAR)', ylabel='true accept rate (TAR)')
axes[1].legend()
plt.tight_layout()
finish(fig, '07_verification')

**Question:** How well can each model tell "same person" from "different people" for people it has never seen?

**Answer:** FaceNet almost perfectly; the generic ImageNet features and Eigenfaces are far behind.

| Model | Test AUC | Equal error rate | Same-person pairs accepted at the threshold |
|---|---|---|---|
| FaceNet | 0.999 | 1.1 % | 96.0 % (with 0.07 % false accepts) |
| ResNet-18 (ImageNet) | 0.776 | 29.3 % | 6.6 % (0.20 % false accepts) |
| Eigenfaces (PCA) | 0.730 | 33.0 % | 10.0 % (0.37 % false accepts) |

- the threshold was chosen on **other people** (validation half) and kept for test, so the test numbers are what a deployment would see
- most of FaceNet's mistakes are same-person pairs with a big change in age, pose, glasses or lighting
- In short → a face-trained embedding plus one number (the threshold) does what the 4-person classifier can't: it works for anyone

# [8/10] Recognising my friends from one photo each, and saying "unknown"

**Can the system recognise my friends and turn away everyone else?**

**What I'm trying to do** → the open-set version of [4/10]: enroll each friend with their **one original photo** (no training at all), then for every new face find the closest friend and accept it only if the similarity passes the threshold from [7/10]. Probes: the friends' test copies and the 1,000 LFW strangers from [5/10].

**For a fair comparison** → I also give the CNN a reject option: "unknown" when its highest probability is below a cut-off that still accepts 95% of the known test copies.

In [ ]:
def facenet_from_faces(faces):
    x = torch.from_numpy((np.stack([np.asarray(f.resize((IMG_SIZE, IMG_SIZE))) for f in faces]).astype(np.float32) - 127.5) / 128).permute(0, 3, 1, 2)
    with torch.no_grad():
        return torch.cat([F.normalize(facenet(x[i:i + BATCH_SIZE].to(DEVICE)), dim=1).cpu() for i in range(0, len(x), BATCH_SIZE)]).numpy()


gallery = facenet_from_faces([friend_faces[n] for n in FRIENDS])           # one embedding per friend, from the original photo
E_known = facenet_from_faces([cls_faces[i] for i in te_idx])
known_true = y_cls[te_idx]
E_real = facenet_from_faces(list(real_ok.face))                            # unseen photos: the real test
E_strangers = np.stack([E_face[i] for i in faces_df.index[faces_df.path.isin(strangers.path)]])
friend_sim = pd.DataFrame(gallery @ gallery.T, index=FRIENDS, columns=FRIENDS)


def open_set(E_probe, threshold):
    sims = E_probe @ gallery.T
    return sims.argmax(1), sims.max(1) >= threshold


pred_k, acc_k = open_set(E_known, THRESHOLD)
pred_r, acc_r = open_set(E_real, THRESHOLD)
_, acc_s = open_set(E_strangers, THRESHOLD)
real_ok['facenet'] = np.where(acc_r, pred_r, -1)                              # -1 = "unknown"
real_ok['best_sim'] = (E_real @ gallery.T).max(1)
cut = np.quantile(p_te.max(1), 0.05)
open_rows = pd.DataFrame({
    'small CNN + probability cut-off': {'enrollment': f'{len(tr_idx)} augmented photos, retrain per person',
                                        'copies named correctly': ((p_te.argmax(1) == y_cls[te_idx]) & (p_te.max(1) >= cut)).mean(),
                                        'unseen photos named correctly': ((p_real.argmax(1) == real_ok.target) & (p_real.max(1) >= cut)).mean(),
                                        'strangers turned away': (p_strangers.max(1) < cut).mean()},
    'FaceNet + 1 photo per person': {'enrollment': '1 photo per person, no training',
                                     'copies named correctly': ((pred_k == known_true) & acc_k).mean(),
                                     'unseen photos named correctly': (real_ok.facenet == real_ok.target).mean(),
                                     'strangers turned away': 1 - acc_s.mean()}}).T
outcome = pd.Series(np.select([real_ok.facenet == real_ok.target, real_ok.facenet == -1], ['right friend', 'unknown'], 'wrong friend'))
by_head = pd.crosstab(real_ok['head'], outcome, normalize='index').reindex(columns=['right friend', 'unknown', 'wrong friend'], fill_value=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={'width_ratios': [1, 1.25]})
sns.heatmap(friend_sim, annot=True, fmt='.2f', cmap='Greys', vmin=0, vmax=1, cbar=False, ax=axes[0], annot_kws={'size': 12})
axes[0].set_title(f'FaceNet similarity between my friends\n(threshold for "same person": {THRESHOLD:.2f})')
for (r, c), v in np.ndenumerate(friend_sim.to_numpy()):
    if r != c and v >= THRESHOLD:
        axes[0].add_patch(plt.Rectangle((c, r), 1, 1, fill=False, ec=RED, lw=3))
own = (E_real * gallery[real_ok.target.to_numpy()]).sum(1)                  # similarity of each unseen photo to the friend it shows
best = np.r_[own, (E_strangers @ gallery.T).max(1)]
who = [f'friend, unseen photo ({h})' for h in real_ok['head']] + ['stranger (LFW)'] * len(E_strangers)
sns.histplot(x=best, hue=who, bins=40, stat='probability', common_norm=False, element='step', ax=axes[1],
             palette={'friend, unseen photo (frontal)': 'black', 'friend, unseen photo (turned)': '#616161',
                      'friend, unseen photo (side)': '#bdbdbd', 'stranger (LFW)': RED})
axes[1].axvline(THRESHOLD, color=RED, lw=2, ls='--', label='threshold')
axes[1].set(title='Unseen photos: similarity to their own enrollment photo; strangers: to the closest friend', xlabel='cosine similarity')
plt.tight_layout()
finish(fig, '08_friends_open_set')

In [ ]:
display(friend_sim.round(2))
display(open_rows)
display(by_head.round(3))
display(pd.crosstab(real_ok.friend.rename('true friend'), real_ok.facenet.map({**dict(enumerate(FRIENDS)), -1: 'unknown'}).rename('FaceNet says')))
show_block('CNN cut-off', {'cut-off (95 % of copies pass)': f'{cut:.4f}',
                           'unseen friend photos above it': f'{(p_real.max(1) >= cut).mean():.1%}',
                           'strangers above it': f'{(p_strangers.max(1) >= cut).mean():.1%}'})

**Question:** How far apart does FaceNet put my four friends, and can it recognise them while turning strangers away?

**Answer:** Yes. Enrolled from one graduation photo each, FaceNet names **95.5 % of the 88 unseen photos** correctly and turns away **99.7 %** of strangers.

| | Copies named correctly | Unseen photos named correctly | Strangers turned away |
|---|---|---|---|
| Small CNN + probability cut-off (600 augmented photos) | 95.0 % | 0 % | 57.0 % |
| FaceNet + 1 photo per friend, threshold from [7/10] | 100 % | 95.5 % | 99.7 % |

- **By head turn** → frontal 98.2 %, turned 94.7 %, side view 85.7 % named correctly; no friend was ever called "unknown"
- **The only mistakes** → 4 of Aum's 10 photos are named "Dong"; Dong (24), Mark (40) and TJ (14) are recognised on every photo
- **Why Dong and Aum** → their enrollment photos are already at similarity **0.57**, above the 0.53 threshold: the same studio, gown, light and pose pull two different people together. Enrolling from a less staged photo, or from 2-3 photos, would separate them
- **The CNN's cut-off** → it accepts 95 % of the copies only above probability 0.9997; **no** unseen photo of my friends gets that high, while 43 % of strangers do. It is more sure about strangers than about my friends
- adding a 5th friend = one more photo in the gallery; no retraining
- In short → recognition = comparing with enrolled photos, with "unknown" as a normal answer; and enrollment photos taken in the same session can look more alike than the people are

# [9/10] Where does the model look?

**Which parts of the face does the "same person" decision rest on?**

**Occlusion test in one line** → cover one small square of the face at a time with grey and see how much the similarity to the other photo drops; a big drop means that square mattered.

**How, in 3 steps**
1. Take a friend's enrolled photo and their most frontal unseen photo
2. Slide a 20 × 20 grey square over the unseen photo, step 10 pixels, and recompute the cosine similarity each time
3. Colour every position by the drop: red = important for the match
- In short → a map of where the identity "lives" for this model

In [ ]:
def face_tensor(face):
    return torch.from_numpy((np.asarray(face.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32)[None] - 127.5) / 128).permute(0, 3, 1, 2)


def occlusion_map(face_a, face_b, patch=20, stride=10):
    with torch.no_grad():
        ea = F.normalize(facenet(face_tensor(face_a).to(DEVICE)), dim=1)
        xb = face_tensor(face_b)
        base = (F.normalize(facenet(xb.to(DEVICE)), dim=1) * ea).sum().item()
        steps = range(0, IMG_SIZE - patch + 1, stride)
        batch = []
        for y in steps:
            for x in steps:
                t = xb.clone()
                t[:, :, y:y + patch, x:x + patch] = 0
                batch.append(t)
        sims = (F.normalize(facenet(torch.cat(batch).to(DEVICE)), dim=1) * ea).sum(1).cpu().numpy()
    heat = np.zeros((IMG_SIZE, IMG_SIZE))
    count = np.zeros_like(heat)
    for (y, x), s in zip([(y, x) for y in steps for x in steps], sims):
        heat[y:y + patch, x:x + patch] += base - s
        count[y:y + patch, x:x + patch] += 1
    return base, heat / np.maximum(count, 1)


fig, axes = plt.subplots(len(FRIENDS), 3, figsize=(10, 3.45 * len(FRIENDS)))
for row, n in enumerate(FRIENDS):
    cand = real_ok[(real_ok.target == row)].sort_values('turn')
    probe = cand.face.iloc[0] if len(cand) else cls_faces[te_idx[y_cls[te_idx] == row][0]]      # most frontal unseen photo
    base, heat = occlusion_map(friend_faces[n], probe)
    axes[row, 0].imshow(friend_faces[n])
    axes[row, 0].set_title(f'enrolled: {n}', fontsize=9)
    axes[row, 1].imshow(probe)
    axes[row, 1].set_title(f'unseen photo (similarity {base:.2f})', fontsize=9)
    axes[row, 2].imshow(probe)
    axes[row, 2].imshow(heat, cmap='Reds', alpha=0.55)
    axes[row, 2].set_title('drop when covered (red = important)', fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Occlusion test on FaceNet', fontweight='bold')
plt.tight_layout()
finish(fig, '09_occlusion')

**Question:** Which parts of the face does the decision rest on?

**Answer:** Mostly the eyes, the nose and the area between the eyebrows (for some faces also the mouth); covering the background, the hair or the cheeks changes the similarity much less.
- a good sign: the model reads the face, not the photo's background or the news agency's style
- covering both eyes alone is often enough to break a match, which is also why sunglasses defeat many systems
- In short → identity sits in the middle of the face

# [10/10] App and summary

**What did I learn, and what should a reader keep in mind?**

**Aim** → save the threshold for the app, optionally start it, and sum up.

In [ ]:
settings = {'threshold': THRESHOLD, 'target_far': TARGET_FAR, 'crop': CROP, 'img_size': IMG_SIZE,
            'test_auc': float(ver_table.loc['FaceNet', 'test AUC']), 'test_tar': float(ver_table.loc['FaceNet', 'test TAR at threshold'])}
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
(Path(WORK_DIR) / 'settings.json').write_text(json.dumps(settings, indent=1))
display(pd.Series(settings, name='app settings'))

In [ ]:
if LAUNCH_APP:
    import gradio as gr

    def verify(img_a, img_b):
        if img_a is None or img_b is None:
            return 'Upload two face photos.'
        tensors = []
        for img in (img_a, img_b):
            face = Image.fromarray(img).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            tensors.append(torch.from_numpy((np.asarray(face, np.float32) - 127.5) / 128).permute(2, 0, 1))
        with torch.no_grad():
            e = F.normalize(facenet(torch.stack(tensors).to(DEVICE)), dim=1)
        s = float((e[0] * e[1]).sum())
        return f"{'Same person' if s >= THRESHOLD else 'Different people'} · similarity {s:.2f} (threshold {THRESHOLD:.2f})"

    gr.Interface(verify, [gr.Image(label='face A'), gr.Image(label='face B')], gr.Textbox(label='result'),
                 title='Face verification (FaceNet + threshold)',
                 description='Upload two cropped face photos. A demo, not an identity check.').launch(share=True)
else:
    print('LAUNCH_APP = False → the app is not started (the standalone version is app.py in the repo)')

**Findings**
- A classifier for my 4 friends scores 100 % on copies of their photos but only 53 % on new photos of them (always saying "Mark" gets 45 %), and it names every stranger as one of them: 88 % become "Mark", 84 % with probability ≥ 0.9
- A face-trained embedding (FaceNet) separates LFW people it has never seen almost perfectly (AUC 0.999, equal error rate 1.1 %); generic ImageNet features (0.78) and Eigenfaces (0.73) don't come close
- Enrolling each friend with one graduation photo, FaceNet names **95.5 %** of 88 new photos correctly (frontal 98 %, side views 86 %) **and** turns away 99.7 % of strangers
- Its only mistakes: 4 of Aum's photos named "Dong", whose enrollment photos are more alike (0.57) than the threshold allows: same studio, same gown
- The decision rests mostly on the eyes, eyebrows and nose

**Limitations and responsible use**
- **Who is in LFW** → news photos of public figures, reported to be mostly men with lighter skin; error rates for other groups can be higher, and LFW has no labels to measure that here
- **Easy setting** → mostly frontal, well-lit, roughly aligned faces; real cameras see harder ones
- **Fixed crop** → no face detector in the notebook; the app expects a cropped face
- **Few unseen photos** → 88 new photos of 4 people (only 10 of Aum) is enough to see the pattern, not to quote precise accuracies; the "same or different" numbers come from 840 unseen LFW people
- **Consent** → faces are biometric data: use public research data or photos of people who agreed to it, never collect faces quietly

# Run on Kaggle
1. Create → **Import Notebook** → upload this file
2. **Add Input** → dataset `jessicali9530/lfw-dataset` (found anywhere under `/kaggle/input`; without it, LFW is downloaded)
3. Settings → Accelerator **GPU T4**, Internet **On** (pip install, FaceNet and ResNet weights)
4. Run All → about 5 minutes on a GPU; my friends' 4 photos are downloaded from the GitHub repository